# 10. Deep Q-Network (DQN)

**課題**: 倒立振子を深層ニューラルネットワークで制御する

前章（07_q_learning.ipynb）では **テーブル型 Q 学習** を使いました。状態を離散化して Q テーブルに保存する方法です。  
本章では Q(s, a) をニューラルネットワークで近似する **Deep Q-Network (DQN)** を実装します。

## DQN の核心アイデア（Mnih et al., 2015）

テーブル型 Q 学習をそのままニューラルネットで置き換えると学習が **発散** しやすい理由：

| 問題 | 原因 | DQN の解決策 |
|------|------|--------------|
| 相関した学習データ | 連続ステップは高度に相関 | **Experience Replay Buffer** |
| TD ターゲットの不安定性 | 更新のたびに目標値が動く | **Target Network** |

### アルゴリズム概要

```
Online  network  Q(s, a; θ)     ← 毎ステップ更新
Target  network  Q(s, a; θ⁻)   ← N ステップごとにθでコピー

TD target:  y = r + γ · max_{a'} Q(s', a'; θ⁻)  (done=False)
            y = r                                 (done=True)

Loss:  L(θ) = E[(y - Q(s, a; θ))²]
```

### テーブル Q 学習との対比

| 観点 | Tabular Q | DQN |
|------|-----------|-----|
| 状態表現 | 離散化インデックス | 連続ベクトルをそのまま |
| Q 値保存 | Q テーブル（配列） | ニューラルネットワーク |
| 汎化 | なし（各状態独立） | あり（似た状態を補間） |
| 状態空間スケール | 小規模 | 大規模・連続 |
| 安定化テクニック | 不要 | リプレイバッファ・ターゲットネット |

## セットアップ

In [ ]:
import sys
import os

# rlpg パッケージを読み込めるようにパスを追加
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'rlpg'))
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from src.environments.pendulum import InvertedPendulumEnv
from src.policies.dqn_policy import ReplayBuffer, QNetwork, DQNPolicy

print('✅ インポート完了')

## 1. Experience Replay Buffer

リプレイバッファは「過去の経験を溜めておく倉庫」です。  
`(state, action_idx, reward, next_state, done)` のタプルを保存し、ランダムにサンプリングして学習します。

**なぜランダムサンプル？**  
連続したステップをそのまま学習すると、`s_t → s_{t+1} → s_{t+2} …` の相関が強くなりすぎます。  
ランダムサンプルで相関を壊すことで学習が安定します。

In [ ]:
# ReplayBuffer の動作確認
env_demo = InvertedPendulumEnv()
buf = ReplayBuffer(capacity=1_000)

state = env_demo.reset()
for _ in range(200):
    action_idx = np.random.randint(11)
    action = np.linspace(-10, 10, 11)[action_idx]
    next_state, reward, done, _ = env_demo.step(action)
    buf.push(state, action_idx, reward, next_state, done)
    state = env_demo.reset() if done else next_state

print(f'Buffer size: {len(buf)}')  # 200
batch = buf.sample(8)
print(f'Sampled batch size: {len(batch)}')
print(f'Sample transition: state={batch[0].state}, action_idx={batch[0].action_idx}, reward={batch[0].reward:.3f}')

## 2. Q ネットワークの構造

DQN の Q ネットワークは状態 **s** を受け取り、全アクションの Q 値を一度に出力します。

```
Input (4)  →  Linear(64) → ReLU → Linear(64) → ReLU → Linear(11)
 [s]                                                    [Q(s,a₀)…Q(s,a₁₀)]
```

テーブル型では `Q_table[state_idx, action_idx]` を lookup していたのに対し、  
ネットワークは連続状態ベクトルを直接受け取り、補間して Q 値を推定します。

In [ ]:
import torch

net = QNetwork(state_dim=4, n_actions=11, hidden_sizes=[64, 64])
print(net.net)

# ランダム状態で Q 値を確認
dummy_state = torch.FloatTensor([[0.0, 0.0, 0.05, 0.0]])  # ほぼ直立
q_vals = net(dummy_state)
print(f'\nQ値 (未学習): {q_vals.detach().numpy().round(3)}')
print(f'最適アクションインデックス: {q_vals.argmax().item()}')

## 3. DQN エージェントの構成

`DQNPolicy` が管理するコンポーネント：

| コンポーネント | 役割 |
|----------------|------|
| `online_net` | 毎ステップ更新される Q ネットワーク |
| `target_net` | TD ターゲット計算用（定期コピー） |
| `buffer` | リプレイバッファ |
| `optimizer` | Adam |

ε は `decay_epsilon()` で指数的に減衰します。

In [ ]:
agent = DQNPolicy(
    n_actions=11,
    hidden_sizes=[64, 64],
    lr=1e-3,
    gamma=0.99,
    epsilon=1.0,
    epsilon_min=0.01,
    epsilon_decay=0.995,
    batch_size=64,
    buffer_capacity=10_000,
    target_update_freq=200,   # 200 ステップごとにターゲットネット更新
)
print(agent)
print(f'\n--- オンラインネット ---')
print(agent.online_net.net)

## 4. 学習ループ

DQN の学習ループは以下の流れです：

```
for episode in range(N):
    state = env.reset()
    while not done:
        action, action_idx = agent.get_action_train(state)   # ε-greedy
        next_state, reward, done, _ = env.step(action)
        agent.push(state, action_idx, reward, next_state, done)  # バッファに追加
        loss = agent.update()                                 # ミニバッチ学習
        state = next_state
    agent.decay_epsilon()                                     # ε 減衰
```

テーブル Q 学習（`update_q` を毎ステップ呼ぶ）と似ていますが、  
「バッファに蓄積してからランダムバッチで学習」が重要な違いです。

In [ ]:
# --- 学習設定 ---
N_EPISODES = 400
MAX_STEPS   = 500
LOG_EVERY   = 20

env = InvertedPendulumEnv(max_steps=MAX_STEPS)

episode_rewards   = []
episode_lengths   = []
epsilon_history   = []
loss_history      = []

for episode in range(1, N_EPISODES + 1):
    state = env.reset()
    total_reward = 0.0
    step_count   = 0
    ep_losses    = []

    done = False
    while not done:
        # ε-greedy 行動選択
        action, action_idx = agent.get_action_train(state)

        # 環境ステップ
        next_state, reward, done, _ = env.step(action)

        # バッファに保存
        agent.push(state, action_idx, reward, next_state, done)

        # ミニバッチ学習
        loss = agent.update()
        if loss is not None:
            ep_losses.append(loss)

        state         = next_state
        total_reward += reward
        step_count   += 1

    # ε 減衰（エピソード終了後）
    agent.decay_epsilon()

    episode_rewards.append(total_reward)
    episode_lengths.append(step_count)
    epsilon_history.append(agent.epsilon)
    loss_history.append(np.mean(ep_losses) if ep_losses else np.nan)

    if episode % LOG_EVERY == 0:
        avg_r = np.mean(episode_rewards[-LOG_EVERY:])
        avg_l = np.mean(episode_lengths[-LOG_EVERY:])
        print(f'Episode {episode:4d} | avg_reward={avg_r:7.1f} | avg_steps={avg_l:5.0f} | ε={agent.epsilon:.3f} | buf={len(agent.buffer)}')

print('\n✅ 学習完了')

## 5. 学習曲線の可視化

4 つのメトリクスをプロットします：

- **エピソード報酬** – 高いほどよい。倒立維持時間に比例
- **エピソード長** – バランス継続ステップ数（最大 500）
- **ε（探索率）** – 指数減衰で 1.0 → 0.01
- **TD ロス** – 学習初期は高く、収束すると下がる

In [ ]:
def smooth(data, window=20):
    """移動平均で曲線を滑らかにする"""
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('DQN 学習曲線 — 倒立振子', fontsize=14, fontweight='bold')

ep = np.arange(1, N_EPISODES + 1)

# ① 報酬
ax = axes[0, 0]
ax.plot(ep, episode_rewards, alpha=0.3, color='royalblue', label='報酬（生）')
sw = min(20, N_EPISODES // 5)
ax.plot(ep[sw-1:], smooth(episode_rewards, sw), color='royalblue', lw=2, label=f'移動平均({sw})')
ax.set_xlabel('エピソード'); ax.set_ylabel('累積報酬')
ax.set_title('エピソード報酬'); ax.legend(); ax.grid(alpha=0.3)

# ② エピソード長
ax = axes[0, 1]
ax.plot(ep, episode_lengths, alpha=0.3, color='darkorange')
ax.plot(ep[sw-1:], smooth(episode_lengths, sw), color='darkorange', lw=2, label=f'移動平均({sw})')
ax.axhline(MAX_STEPS, ls='--', color='gray', alpha=0.7, label=f'最大 {MAX_STEPS} ステップ')
ax.set_xlabel('エピソード'); ax.set_ylabel('ステップ数')
ax.set_title('エピソード長'); ax.legend(); ax.grid(alpha=0.3)

# ③ ε
ax = axes[1, 0]
ax.plot(ep, epsilon_history, color='forestgreen', lw=2)
ax.set_xlabel('エピソード'); ax.set_ylabel('ε')
ax.set_title('探索率 ε の減衰'); ax.grid(alpha=0.3)

# ④ TD ロス
ax = axes[1, 1]
valid = [(i, l) for i, l in enumerate(loss_history) if not np.isnan(l)]
if valid:
    xi, yi = zip(*valid)
    ax.plot(np.array(xi)+1, yi, alpha=0.4, color='crimson')
    sw2 = min(20, len(yi) // 3)
    if sw2 > 1:
        ax.plot(np.array(xi[sw2-1:])+1, smooth(list(yi), sw2), color='crimson', lw=2, label=f'移動平均({sw2})')
    ax.legend()
ax.set_xlabel('エピソード'); ax.set_ylabel('平均 TD ロス')
ax.set_title('TD ロス'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 学習済みエージェントの評価

探索なし（ε = 0 の greedy 方策）で評価します。

In [ ]:
def evaluate_dqn(policy: DQNPolicy, n_episodes: int = 10, max_steps: int = 500) -> dict:
    """学習済み DQN を greedy 方策で評価する"""
    env_eval = InvertedPendulumEnv(max_steps=max_steps)
    rewards = []
    lengths = []

    for _ in range(n_episodes):
        state = env_eval.reset()
        total_r, steps, done = 0.0, 0, False
        while not done:
            action = policy.get_action(state)   # greedy（ε なし）
            state, r, done, _ = env_eval.step(action)
            total_r += r
            steps   += 1
        rewards.append(total_r)
        lengths.append(steps)

    return {
        'mean_reward' : np.mean(rewards),
        'std_reward'  : np.std(rewards),
        'mean_length' : np.mean(lengths),
        'max_steps'   : max_steps,
        'solved_rate' : np.mean([l >= max_steps for l in lengths]),
    }

result = evaluate_dqn(agent, n_episodes=10)
print('=== 評価結果 (greedy, 10 エピソード) ===')
print(f'  平均報酬    : {result["mean_reward"]:8.1f} ± {result["std_reward"]:.1f}')
print(f'  平均ステップ: {result["mean_length"]:8.1f} / {result["max_steps"]}')
print(f'  完走率      : {result["solved_rate"]*100:.0f}%  (={result["max_steps"]}ステップ完走)')

## 7. Q 値の可視化

学習済みネットワークが「どんな状態でどのアクションを好むか」を可視化します。  
角度 θ と角速度 θ̇ を変化させたときのグリーディアクションを確認します。

In [ ]:
theta_range      = np.linspace(-0.3, 0.3, 40)    # 角度 ±0.3 rad ≈ ±17°
theta_dot_range  = np.linspace(-2.0, 2.0, 40)    # 角速度
action_values    = np.linspace(-10, 10, 11)       # 連続アクション値

greedy_actions = np.zeros((40, 40))

for i, th in enumerate(theta_range):
    for j, td in enumerate(theta_dot_range):
        state = np.array([0.0, 0.0, th, td], dtype=np.float32)
        action = agent.get_action(state)           # greedy
        greedy_actions[i, j] = action

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    greedy_actions,
    origin='lower',
    extent=[theta_dot_range[0], theta_dot_range[-1], theta_range[0], theta_range[-1]],
    aspect='auto',
    cmap='RdBu',
    vmin=-10, vmax=10,
)
plt.colorbar(im, ax=ax, label='グリーディアクション [N]')
ax.set_xlabel('角速度 θ̇ [rad/s]')
ax.set_ylabel('角度 θ [rad]')
ax.set_title('学習済み DQN のグリーディアクション\n(x=0, ẋ=0 固定)')
ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.7)
ax.axvline(0, color='white', lw=0.8, ls='--', alpha=0.7)
plt.tight_layout()
plt.show()

print('解釈ヒント:')
print('  右上（θ>0, θ̇>0）= 右に倒れかけ → 青（正の力）で右に押す')
print('  左下（θ<0, θ̇<0）= 左に倒れかけ → 赤（負の力）で左に押す')

## 8. テーブル Q 学習との比較

同じ環境でテーブル型 Q 学習（07章）と比較します。

In [ ]:
from src.policies.q_policy import QPolicy
from src.utils.discretizer import StateDiscretizer

# --- テーブル Q 学習 (短縮版) ---
disc = StateDiscretizer()
q_agent = QPolicy(
    state_discretizer=disc,
    n_actions=11,
    alpha=0.1,
    gamma=0.99,
    epsilon=1.0,
    epsilon_min=0.01,
    epsilon_decay=0.995,
)

env_q   = InvertedPendulumEnv(max_steps=500)
q_rewards = []

for episode in range(N_EPISODES):
    state = env_q.reset()
    total_r, done = 0.0, False
    while not done:
        action = q_agent.get_action(state)
        ns, r, done, _ = env_q.step(action)
        q_agent.update_q(state, action, r, ns, done)
        state, total_r = ns, total_r + r
    q_agent.decay_epsilon()
    q_rewards.append(total_r)

print('テーブル Q 学習 学習完了')

# --- 比較プロット ---
sw = 20
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(ep[sw-1:], smooth(episode_rewards, sw), label='DQN (深層)', lw=2, color='royalblue')
ax.plot(ep[sw-1:], smooth(q_rewards, sw),       label='テーブル Q (離散化)', lw=2, color='darkorange', ls='--')

ax.set_xlabel('エピソード')
ax.set_ylabel(f'報酬（移動平均 {sw}）')
ax.set_title('DQN vs テーブル Q 学習 — 学習曲線比較')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. まとめ

### 本章で学んだこと

| 概念 | 説明 |
|------|------|
| **Q ネットワーク** | Q(s,a) を近似するニューラルネット。全アクションを一度に出力 |
| **Experience Replay** | 過去の経験を保存してランダムサンプリング。相関を壊して安定化 |
| **Target Network** | TD ターゲット計算に使う「遅れて更新されるコピー」。学習を安定させる |
| **ε-greedy 減衰** | 探索から活用へ段階的に移行 |
| **Gradient Clipping** | 大きな勾配を抑制してネットを壊さないように保護 |

### テーブル Q 学習との実装差分

```python
# テーブル Q 学習
td_error = target - q_table[s, a]
q_table[s, a] += alpha * td_error  # 直接更新

# DQN
buffer.push(s, a, r, s', done)     # 先にバッファへ
batch = buffer.sample(64)          # ランダムサンプル
y = r + γ * target_net(s').max()   # ターゲットネットで TD target
loss = mse(online_net(s)[a], y)    # ネットのパラメータを更新
```

### 次のステップ（11章: PPO）

DQN は **離散行動空間** に適しています。  
連続行動空間や複雑なタスクには **PPO (Proximal Policy Optimization)** が現在の標準手法です。  
次章では既存の Actor-Critic（09章）を発展させ、PPO を実装します。